In [2]:
import pandas as pd
import re
import json

In [12]:
# summaries = pd.read_csv('/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/final/nacc/train_with_summary/qwen25_72B_etpr_cog_mci_park.csv')
# summaries = pd.read_csv('/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/final/nacc/train_with_summary/qwen3_14B_etpr_cog_mci_park_nocop.csv')
# summaries = pd.read_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/fm_adrd/adrd_simplified_evaluation/benchmarks/adrd_cog_status_summary/COG_STATUS_test.csv")
# summaries = pd.read_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/fm_adrd/adrd_simplified_evaluation/benchmarks/adrd_neuropath_summary/NP_MAJOR_test.csv")
# summaries = pd.read_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/training_data_grpo/train_summary.csv")
summaries = pd.read_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/training_data_grpo/train_summary_32B.csv")
# summaries = pd.read_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/training_data_grpo/test_summary_30B_A3B.csv")

In [13]:
len(summaries)

36000

In [14]:
import pandas as pd
import re
from collections import Counter

def find_repeated_sentences_in_text(text):
    # Split into sentences using punctuation (adjust if needed)
    sentences = re.split(r'(?<=[.!?])\s+', str(text).strip())
    normalized = [s.strip().lower() for s in sentences if s.strip()]
    counter = Counter(normalized)
    repeated = {sent: count for sent, count in counter.items() if count > 1}
    return repeated

# Load your dataframe
# df = pd.read_csv("your_file.csv")  # Example
# Example dataframe:
# df = pd.DataFrame({'visit_summary': [...texts...]})

repeats_info = []

for idx, row in summaries.iterrows():
    text = row['visit_summary']
    repeats = find_repeated_sentences_in_text(text)
    if repeats:
        for sentence, count in repeats.items():
            repeats_info.append({
                "row_index": idx,
                "sentence": sentence,
                "count": count
            })

# Convert results to a DataFrame for easy analysis
repeats_df = pd.DataFrame(repeats_info)


In [15]:
repeats_df

,row_index,sentence,count
0,6433,"the patient has hyperton, hyperton, and hyperton.",120
1,17452,she does not wear a hearing aid and has impair...,2
2,17452,she uses corrective lenses and has normal visi...,2


In [40]:
# print(summaries.iloc[200]['visit_summary'])

In [41]:
# print(summaries1.iloc[200]['visit_summary'])

In [7]:
repeat_list = set(repeats_df['row_index'])

In [8]:
repeat_list

{2461, 3532}

In [9]:
summaries_dropped_index = summaries.drop(repeat_list).reset_index(drop=True)

In [100]:
# summaries_dropped_index.to_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/final/nacc/train_with_summary/qwen3_14B_etpr_cog_mci_park_nocop_corrected.csv", index=False)

In [129]:
summaries_dropped_index = summaries

In [10]:
ages = []
for i, row in summaries_dropped_index.iterrows():
    pat = json.loads(row['patient_summary'])
    ages.append(pat['Subject Demographics']["Subject's age at visit"])

In [11]:
index = -1
incorrect_age_count = []
incorrect_cop_age_count = []

# Define regex pattern to extract and replace "a/an n-year-old"
remove_pattern = r"\b(\d{1,3})-year-old\s\b"

def remove_incorrect_age(text):
    global count
    global index
    global incorrect_cop_age
    index += 1
    # Process text line by line
    modified_text = []
    for sentence in text.split("."):
        match = re.search(remove_pattern, sentence)
        if match:  # Check if the sentence matches the pattern
            # print(sentence)
            age = match.group(1)  # Extract age
            # if "co-participant" in sentence.lower() or "spouse" in sentence.lower() or "child" in sentence.lower() or "daughter" in sentence.lower() or "friend" in sentence.lower() or "son" in sentence.lower() or "sibling" in sentence.lower() or "paid caregiver" in sentence.lower() or "relative" in sentence.lower():
            #     incorrect_cop_age_count.append((index, age))
                
            # else:   
            #     if int(age) != int(ages[index]):
            #         incorrect_age_count.append((index, age))
                    
            if int(age) != int(ages[index]):
                # print(text)
                incorrect_age_count.append((index, age, match))
            
    
    # return final_text

summaries_dropped_index["visit_summary"].apply(remove_incorrect_age)
print("Total sentences with incorrect co participant age:", len(incorrect_cop_age_count))
print("Total sentences with incorrect age:", len(incorrect_age_count))

Total sentences with incorrect co participant age: 0
Total sentences with incorrect age: 0


In [47]:
incorrect_age_count

[(31342, '40', <re.Match object; span=(34, 46), match='40-year-old '>),
 (31891, '35', <re.Match object; span=(80, 92), match='35-year-old '>)]

In [51]:
ages[31891]

74

In [53]:
print(summaries_dropped_index.iloc[31891]['visit_summary'])
# print(summaries_dropped_index.iloc[31891]['patient_summary'])

The patient is a 74-year-old right-handed Black or African American woman who lives independently in a private residence and is currently divorced. She has completed 16 years of education and is fluent in English. Her primary reason for visiting the Alzheimer’s Disease Center is to participate in a research study, and she was referred by an unspecified source. The patient is taking eight medications, including diabetes and antihypertensive agents, a calcium channel blocker, and an antidepressant, with palbociclib being the specific medication reported within two weeks of the visit. There is no reported family history of cognitive impairment among first-degree relatives.

On the Neuropsychiatric Inventory Questionnaire, completed with the help of a 35-year-old friend, there are no reported behavioral or psychological symptoms in the last month, including no signs of depression, anxiety, agitation, or hallucinations. The Geriatric Depression Scale indicates a score of 3 out of 15, with t

The patient is a 65-year-old right-handed Black or African American woman who lives independently in a private residence and is currently separated. She has completed 18 years of education and is fluent in English. Her primary motivation for participating in the Alzheimer's Disease Center (ADC) is to contribute to a research study, and she was referred by a non-professional contact. She reports no history of cigarette use and consumes alcohol a few times a week, though she does not meet criteria for alcohol abuse. She is taking seven medications, including an angiotensin II inhibitor, a calcium channel blocker, and an antidepressant, with FLUTICASONE-SALMETEROL listed among the medications used within two weeks of the visit. Her mother has a reported history of cognitive impairment, but no such history was reported for her father.

The patient’s physical examination reveals a body mass index (BMI) of 29.4, indicating she is overweight. Her resting heart rate is 68 beats per minute, and her blood pressure is 144/98 mmHg. She wears corrective lenses for vision, which are effective in restoring functional vision, and she has normal hearing without the use of hearing aids. Her medical history includes active hypertension and hypercholesterolemia, as well as osteoarthritis affecting the lower extremities and a recent diagnosis of urinary incontinence. She has a history of depression both in the last two years and beyond, but no current anxiety. Additionally, she has a recent/active diagnosis of post-traumatic stress disorder (PTSD), though no history of traumatic brain injury, stroke, or Parkinsonian disorders.

On the Geriatric Depression Scale (GDS), the patient scored 12 out of 15, indicating significant depressive symptoms. She reports feeling worthless, helpless, hopeless, and experiencing memory concerns, while also expressing satisfaction with her life despite these feelings. Her responses also reflect a preference to remain at home and a tendency to feel bored and have dropped many of her usual activities. However, she is still able to complete the GDS, and no cognitive or physical limitations were noted that would prevent her from doing so.

The Functional Assessment Scale (FAS) indicates that the patient maintains independence in all domains assessed, including paying attention, managing finances, preparing meals, shopping, and remembering appointments and medications. The Neuropsychiatric Inventory Questionnaire (NPI-Q) reveals no behavioral or psychiatric symptoms in the past month, including no delusions, agitation, depression, anxiety, or hallucinations. The informant for the NPI-Q is a 40-year-old friend, suggesting the patient has a supportive social network.

The neuropsychological battery summary scores show a range of performance. On the Montreal Cognitive Assessment (MoCA), the patient scored 23 out of 30, corrected for education, with intact orientation, attention, and language skills but with some deficits in delayed recall and abstraction. On the Craft Story 21 Recall, she recalled 14 units using paraphrase scoring and 15 units using verbatim scoring for the delayed recall, with 20 units recalled immediately. The Benson Figure was copied with a score of 14 out of 17, and the delayed drawing scored 8 out of 17, indicating some visuospatial memory impairment. The Trail Making Test showed normal performance on Part A (33 seconds) and only mild slowing on Part B (97 seconds). The Multilingual Naming Test (MINT) yielded a total score of 27 out of 32, with only minor difficulties requiring semantic or phonemic cues. The Number Span Test showed a forward span of 7 and a backward span of 5. The clinician noted that three or more scores were abnormal or lower than expected, suggesting subtle cognitive impairments.

In [141]:
import re

# Define regex pattern to identify relevant sentences
pattern = r"\b\d{1,3}-year-old individual\b.*?\bsmoking\b|\bsmoking\b.*?\b\d{1,3}-year-old\b"

# Define regex pattern to extract and replace "a/an n-year-old"
remove_pattern = r"\b(\d{1,3})-year-old\s\b"

# Regex pattern to match "The patient quit smoking at the age of X"
quit_smoking_pattern = r"The patient quit smoking at the age of (\d+)"


index = 0
incorrect_smoking_age_count = []
invalid_smoking_age_count = []
quit_30_days_count = []

def clean_smoking_part(text):
    global count
    global index
    global invalid_smoking_age
    global quit_30_days_count
    
    modified_text = []
    
    for sentence in text.split("."):
        # sentence = sentence.strip()  # Remove leading/trailing spaces
        
        if "They quit smoking 30 days prior to the initial visit" in sentence:
            # sentence = sentence.replace("They quit smoking 30 days prior to the initial visit", "They did not smoke cigarettes for 30 days prior to the visit")
            quit_30_days_count.append(index)
        
        # Check for "The patient quit smoking at the age of X"
        quit_match = re.search(quit_smoking_pattern, sentence)
        if quit_match:
            age = int(quit_match.group(1))
            if int(age) > int(ages[index]):  # Skip this sentence if age < 60
                invalid_smoking_age_count.append(index)
                continue

        # Check for "n-year-old" + "smoking"
        match = re.search(remove_pattern, sentence)
        # if match and re.search(pattern, sentence):  
        if match and "smoking" in sentence.lower(): 
            age = match.group(1)  # Extract age
            if int(age) != int(ages[index]):
                # sentence = re.sub(remove_pattern, "", sentence).strip()   # Replace "n-year-old" with ""
                incorrect_smoking_age_count.append(index)
            
            if int(age) <= int(ages[index]) and (int(age) == int(smoke_quit_age[index]) and smoke_quit_age[index] != -1):
                # new_sentence = f"\n\nThe patient quit smoking at the age of {age}. "  # Create new sentence
                # modified_text.append(new_sentence + sentence)  # Add new sentence before original
                incorrect_smoking_age_count.append(index)
            # else:
            #     modified_text.append("\n\n" + sentence)
                # print(index)
                # raise ValueError
        # else:
        #     modified_text.append(sentence)  # Keep other sentences unchanged

    # Join sentences back into a full text
    # final_text = ".".join(modified_text)  # Removes any accidental empty sentences
    index += 1
    # return final_text

# Apply function to DataFrame column
summaries_dropped_index["visit_summary"].apply(clean_smoking_part)

print("Total sentences modified:", len(incorrect_smoking_age_count))
print("Invalid smoking age:", len(invalid_smoking_age_count))
print("Invalid quit_smoking_days:", len(quit_30_days_count))


Total sentences modified: 0
Invalid smoking age: 0
Invalid quit_smoking_days: 0
